# Scraping Masters Shot Videos using our Previous Participants

Importing appropriate Libraries

In [7]:
import requests
import time
import re
import asyncio
import pandas as pd
import numpy as numpy
from playwright.async_api import async_playwright

Load in the masters past participants data

In [10]:
masters_players_df = pd.read_csv("C:/Personal Projects/finalized_masters_participants.csv")

masters_players_df.head(10)

,Year,Pos,Name,R1,R2,R3,R4,Total Score,Total Par,Made_Cut
0,2025,1,R McIlroy,72.0,66.0,66.0,73.0,277.0,-11.0,1
1,2025,2,J Rose,65.0,71.0,75.0,66.0,277.0,-11.0,1
2,2025,3,P Reed,71.0,70.0,69.0,69.0,279.0,-9.0,1
3,2025,4,S Scheffler,68.0,71.0,72.0,69.0,280.0,-8.0,1
4,2025,5,S Im,71.0,70.0,71.0,69.0,281.0,-7.0,1
5,2025,5,B DeChambeau,69.0,68.0,69.0,75.0,281.0,-7.0,1
6,2025,7,L Åberg,68.0,73.0,69.0,72.0,282.0,-6.0,1
7,2025,8,X Schauffele,73.0,69.0,70.0,71.0,283.0,-5.0,1
8,2025,8,Z Johnson,72.0,74.0,66.0,71.0,283.0,-5.0,1
9,2025,8,J Day,70.0,70.0,71.0,72.0,283.0,-5.0,1


Now going to the "Master's Vault" we can see that there is only footage for final round shots from 1968 onwards
This means we need to filter out all data prior to 1968 from our table

In [11]:
vault_players = masters_players_df[masters_players_df["Year"]>=1968]

vault_players["Year"].min

<bound method Series.min of 0       2025
1       2025
2       2025
3       2025
4       2025
        ... 
7198    1994
7199    1994
7200    1994
7201    1994
7202    1994
Name: Year, Length: 5076, dtype: int64>

Perfect! It looks like we've dropped all players before 1968. Now what we want to do is create a new loop:
where for i in each row (players), we go to the vault webpage: https://www.masters.com/vault
Once there we will iterate a vault video search through each player, each year if applicable for that player, and each hole that year for said player.
I.e (Tiger Woods 1968 Hole 1-Tiger Woods 2025 Hole 18), and scrape the associated video which comes up and append it into a dataframe of videos
consisting of four columns: player, year, hole #, and the video itself. 

In [ ]:
BASE_URL = "https://www.masters.com/en_US/tournament/past_winners.html"


In [ ]:

url = "https://www.masters.com/apps/vault/api/search"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.masters.com",
    "Referer": "https://www.masters.com/vault/results.html",
    "User-Agent": "Mozilla/5.0",
    "Authorization": "Bearer YOUR_TOKEN_HERE"
}

payload = {
    "query": "rory mcilroy 2025 hole 1"
}

response = requests.post(url, json=payload, headers=headers, timeout=10)

print(response.status_code)
print(response.text[:500])

200
{"docs":[{"id":"final_2025.mp4-2025-9853-9883","round_number":"round 4","scenesummarizationendtime":9883,"video_year":"2025","cmsid":"masters_v1_17445933286885392_en","clip_title":"Rory McIlroy, 2025, Hole No. 1, Shot 6","full_name":"Rory McIlroy","clip_subtitle":"2025 Final Round Broadcast","scenesummarizationstarttime":9853,"end_zone":null,"is_moment":true,"_score":139.532295,"_field_scores":{"video_year":49.955777999999995,"moment_text":7.2738423,"hole_number":40.679418,"mandatory_keyword_con


In [ ]:

data = response.json()

videos = []

for item in data.get("docs", []):
    title = item.get("clip_title")
    player = item.get("full_name")
    year = item.get("video_year")
    vid_id = item.get("id")

    # extract hole number from title
    hole_match = re.search(r'Hole No\. (\d+)', title)
    hole = int(hole_match.group(1)) if hole_match else None

    videos.append({
        "player": player,
        "year": int(year),
        "hole": hole,
        "video_id": vid_id
    })

print(videos)

[{'player': 'Rory McIlroy', 'year': 2025, 'hole': 1, 'video_id': 'final_2025.mp4-2025-9853-9883'}, {'player': 'Rory McIlroy', 'year': 2025, 'hole': 1, 'video_id': 'final_2025.mp4-2025-9765-9798'}, {'player': 'Rory McIlroy', 'year': 2025, 'hole': 1, 'video_id': 'final_2025.mp4-2025-9037-9067'}, {'player': 'Rory McIlroy', 'year': 2025, 'hole': 1, 'video_id': 'final_2025.mp4-2025-9571-9600'}, {'player': 'Rory McIlroy', 'year': 2025, 'hole': 1, 'video_id': 'final_2025.mp4-2025-9389-9438'}, {'player': 'Rory McIlroy', 'year': 2025, 'hole': 1, 'video_id': 'final_2025.mp4-2025-9831-9850'}]


In [13]:

url = "https://www.masters.com/apps/vault/api/search"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.masters.com",
    "Referer": "https://www.masters.com/vault/results.html",
    "User-Agent": "Mozilla/5.0",
    "Authorization": "Bearer YOUR_TOKEN_HERE"
}

results = []

for _, row in vault_players.iterrows():
    player = row["Name"]
    year = row["Year"]

    # Only include players who played Round 4
    if pd.notna(row["R4"]):

        for hole in range(1, 19):

            query = f"{player} {year} hole {hole}"
            payload = {"query": query}

            try:
                response = requests.post(
                    url,
                    json=payload,
                    headers=headers,
                    timeout=10
                )

                if response.status_code != 200:
                    continue

                data = response.json()

                for item in data.get("docs", []):
                    title = item.get("clip_title", "")
                    vid_id = item.get("id")
                    full_name = item.get("full_name")
                    vid_year = item.get("video_year")

                    # Extract hole + shot
                    hole_match = re.search(r'Hole No\. (\d+)', title)
                    shot_match = re.search(r'Shot (\d+)', title)

                    hole_num = int(hole_match.group(1)) if hole_match else None
                    shot_num = int(shot_match.group(1)) if shot_match else None

                    results.append({
                        "player": full_name,
                        "year": int(vid_year) if vid_year else year,
                        "hole": hole_num,
                        "shot": shot_num,
                        "video_id": vid_id
                    })

                time.sleep(0.3)  # avoid rate limiting

            except Exception as e:
                print(f"Error with {player} {year} hole {hole}: {e}")
                continue

df_videos = pd.DataFrame(results).drop_duplicates()

Error with C Schwartzel 1995 hole 4: HTTPSConnectionPool(host='www.masters.com', port=443): Read timed out. (read timeout=10)
Error with C Schwartzel 1995 hole 5: HTTPSConnectionPool(host='www.masters.com', port=443): Read timed out. (read timeout=10)
Error with R Moore 2017 hole 5: HTTPSConnectionPool(host='www.masters.com', port=443): Max retries exceeded with url: /apps/vault/api/search (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000002336DAEA5D0>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))
Error with R Moore 2017 hole 10: HTTPSConnectionPool(host='www.masters.com', port=443): Read timed out. (read timeout=10)
Error with J. Rose 2003 hole 16: HTTPSConnectionPool(host='www.masters.com', port=443): Max retries exceeded with url: /apps/vault/api/search (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000002336DAEA350>, 'Connection to www.masters.com

KeyboardInterrupt: 

In [ ]:
async def scrape_videos(page):
    videos = []

    for _, row in vault_players.iterrows():
        player = row["Player"]
        year = row["Year"]

        if not pd.isna(row["R4"]):
            for hole in range(1, 19):
                shot = 1

                while True:
                    query = f"{player} {year} hole {hole} shot {shot}"
                    print(f"Searching: {query}")

                    # --- Navigate / trigger search ---
                    await page.goto("https://www.masters.com/en_US/video/vault.html")
                    await page.wait_for_load_state("networkidle")

                    # Type into search box (update selector to match real site)
                    search_box = await page.wait_for_selector("input[type='search'], input[placeholder*='Search'], .vault-search-input")
                    await search_box.fill("")
                    await search_box.type(query)
                    await search_box.press("Enter")

                    await page.wait_for_load_state("networkidle")
                    await page.wait_for_timeout(1500)  # let results render

                    # --- Check if any video results exist ---
                    results = await page.query_selector_all(".video-result-item, .vault-video-card, [class*='video']")  # update selector

                    if not results:
                        print(f"  No results for shot {shot} — moving to hole {hole + 1}")
                        break  # no more shots for this hole

                    # --- Grab video metadata from first result ---
                    first = results[0]
                    title = await first.inner_text()
                    link_el = await first.query_selector("a")
                    href = await link_el.get_attribute("href") if link_el else None

                    videos.append({
                        "player": player,
                        "year": year,
                        "hole": hole,
                        "shot": shot,
                        "query": query,
                        "title": title.strip(),
                        "url": href,
                    })

                    print(f"  ✓ Found: {title.strip()}")
                    shot += 1  # try next shot

    return videos